# 03 Build Vocab

Цель этапа: построить стабильные словари токенов по нормализованным событиям.

Вход:

- `data/interim/events_normalized_*.parquet`
- `artifacts/manifests/normalize_manifest.json`

Выход:

- `artifacts/vocab/event_token_vocab.json`
- `artifacts/vocab/json_key_vocab.json`
- `artifacts/vocab/json_path_vocab.json`
- `artifacts/vocab/attribute_vocab.json`
- `artifacts/vocab/numeric_bucket_vocab.json`
- `artifacts/*_frequencies.json`
- `artifacts/manifests/vocab_manifest.json`

In [2]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))
PROJECT_ROOT

PosixPath('/root/projects/bert4rec')

In [3]:
from src.tokenization import build_vocabs_from_normalized_shards, load_manifest, load_vocab_config

config = load_vocab_config(PROJECT_ROOT / "configs" / "vocab.yaml")
normalize_manifest = load_manifest(PROJECT_ROOT / config.input_manifest_path)

print("normalized shards:", normalize_manifest["num_shards"])
print("normalized rows:", normalize_manifest["rows"])
print("parse errors:", normalize_manifest["parse_errors"])
config

normalized shards: 74
normalized rows: 73063802
parse errors: 6708


VocabConfig(input_manifest_path='artifacts/manifests/normalize_manifest.json', vocab_dir='artifacts/vocab', manifest_path='artifacts/manifests/vocab_manifest.json', special_tokens=('[PAD]', '[UNK]', '[MASK]', '[CLS]'), sources={'event_token': 'event_signature', 'json_key': 'json_top_keys', 'json_path': 'json_leaf_paths', 'attribute': 'attribute_tokens', 'numeric_bucket': 'numeric_bucket_tokens'}, limits={'event_token': VocabLimit(min_freq=2, max_size=500000), 'json_key': VocabLimit(min_freq=2, max_size=200000), 'json_path': VocabLimit(min_freq=2, max_size=300000), 'attribute': VocabLimit(min_freq=3, max_size=1000000), 'numeric_bucket': VocabLimit(min_freq=2, max_size=500000)})

## Control Cell

Сначала выполните `DRY_RUN = True`: словари будут построены на первых нескольких shards, чтобы проверить скорость и размеры. Для полного словаря поставьте `DRY_RUN = False`.

In [4]:
DRY_RUN = False
MAX_SHARDS = 3 if DRY_RUN else None

print("DRY_RUN:", DRY_RUN)
print("MAX_SHARDS:", MAX_SHARDS)

DRY_RUN: False
MAX_SHARDS: None


In [5]:
manifest = build_vocabs_from_normalized_shards(
    config=config,
    project_root=PROJECT_ROOT,
    max_shards=MAX_SHARDS,
)

print("num_shards:", manifest["num_shards"])
print("rows_seen:", manifest["rows_seen"])
print("parse_errors_seen:", manifest["parse_errors_seen"])
list(manifest["vocabs"].keys())

Build vocab:   0%|          | 0/74 [00:00<?, ?it/s]

num_shards: 74
rows_seen: 73063802
parse_errors_seen: 6708


['event_token', 'json_key', 'json_path', 'attribute', 'numeric_bucket']

In [6]:
import pandas as pd

summary_rows = []
for name, stats in manifest["vocabs"].items():
    summary_rows.append({
        "vocab": name,
        "size": stats["size"],
        "total_unique": stats["total_unique"],
        "kept_unique": stats["kept_unique"],
        "oov_unique": stats["oov_unique"],
        "coverage_occurrences": stats["coverage_occurrences"],
        "path": stats["vocab_path"],
    })

pd.DataFrame(summary_rows).sort_values("vocab")

,vocab,size,total_unique,kept_unique,oov_unique,coverage_occurrences,path
3,attribute,254821,4496377,254817,4241560,0.991490,artifacts/vocab/attribute_vocab.json
0,event_token,114159,227402,114155,113247,0.998450,artifacts/vocab/event_token_vocab.json
1,json_key,64682,115137,64678,50459,0.999706,artifacts/vocab/json_key_vocab.json
2,json_path,280801,2923974,280797,2643177,0.994548,artifacts/vocab/json_path_vocab.json
4,numeric_bucket,215317,1562403,215313,1347090,0.997040,artifacts/vocab/numeric_bucket_vocab.json


In [ ]:
for name, stats in manifest["vocabs"].items():
    print("\n", name)
    for item in stats["top_tokens"][:10]:
        print(f"  {item['count']:>10}  {item['token']}")

In [ ]:
manifest_path = PROJECT_ROOT / config.manifest_path
print(f"Manifest written to: {manifest_path}")
print(f"Vocab dir: {PROJECT_ROOT / config.vocab_dir}")